# Augustine Qwen3 14B Fine-Tuning with Unsloth (4-bit QLoRA)

**Base Model:** Qwen3 14B

**Dataset:** Augustine ShareGPT JSONL — multi-turn conversations with 4 voice modes (confessional, philosophical, polemical, devotional)

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Deployment Target:** A5000 GPU server via vLLM (24GB VRAM — 14B 4-bit ~8GB fits comfortably)

**Chat Template:** `<|im_start|>role\ncontent<|im_end|>` (ChatML)

**Architecture:** Single-persona LoRA that teaches the model to speak as Augustine across four distinct voice registers. The system prompt varies per voice mode — confessional (Confessions), philosophical (City of God), polemical (Donatist Controversy), devotional (Soliloquies) — so the model learns "when the prompt says Confessions mode, pray like Augustine; when it says City of God mode, argue like Augustine."

## 1. Configuration

All paths and variables for easy configuration.

In [10]:

# ============================================================================
# CONFIGURATION — Augustine Qwen3 14B 4-bit QLoRA
# ============================================================================
# Training: DGX Spark (128GB unified memory)
# Deployment: A5000 GPU server (24GB VRAM) via vLLM (14B 4-bit ~8GB)

# =========================== PATHS (all cascade from PROJECT_ROOT) ===========================
PROJECT_ROOT = "/home/spark/projects/training/biblical"
DATA_DIR = f"{PROJECT_ROOT}/data"
OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

# =========================== MODEL CONFIGURATION ===========================
BASE_LLM = "unsloth/Qwen3-14B-unsloth-bnb-4bit"
MODEL_NAME_BASE = "augustine_qwen3_14b_unsloth_4bit"

# =========================== INPUT DATA ===========================
# Multi-turn ShareGPT JSONL from the Augustine datagen notebook
# Already quality-filtered, multi-turn (4 QA pairs per conversation), grouped by topic
# System prompts vary by voice mode (confessional, philosophical, polemical, devotional)
INPUT_DATA_FILE = f"{DATA_DIR}/training-data/augustine_persona/augustine_sharegpt.jsonl"

# =========================== VOICE MODES ===========================
# Augustine speaks in 4 registers depending on which work the data was drawn from.
# The actual system prompts are EXTRACTED from the JSONL at load time (see data loading cell).
# This dict provides detection phrases and test prompts for each mode.
VOICE_MODES = {
    "confessional": {
        "detection_phrase": "Speaking from the Confessions",
        "description": "Raw autobiographical prayer, the restless heart, the journey from sin to grace",
        "test_prompt": "What was happening in your heart during those years of wandering before your conversion?",
    },
    "philosophical": {
        "detection_phrase": "Speaking from the City of God",
        "description": "Grand theological argument about the two cities, faith & reason, divine providence",
        "test_prompt": "How do you answer the pagan philosophers who say Rome fell because you abandoned their gods?",
    },
    "polemical": {
        "detection_phrase": "Speaking from the Donatist controversy",
        "description": "Sharp rhetorical defense of church unity, validity of sacraments",
        "test_prompt": "Why do you insist that sacraments administered by unworthy priests remain valid?",
    },
    "devotional": {
        "detection_phrase": "Speaking from the Soliloquies",
        "description": "Intimate dialogue between Augustine and Reason, truth, the soul, immortality",
        "test_prompt": "What is the relationship between knowing yourself and knowing God?",
    },
}

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"

# =========================== TRAINING HYPERPARAMETERS ===========================
# MUST be 4096 — conversations average ~2,362 tokens.  At 2048 they get truncated
# and packing can't combine anything.  At 4096, packing fits ~1.7 convos/sequence.
MAX_SEQ_LENGTH = 4096
BATCH_SIZE = 2              # GB10 unified memory prefers smaller batches
GRAD_ACCUM = 4              # Effective batch = 8
LEARNING_RATE = 1e-4
TARGET_EPOCHS = 1

# =========================== LoRA CONFIGURATION ===========================
# r=16, alpha=32: moderate rank — Augustine has rich multi-mode data
# Balanced target modules: attention projections capture voice register switching
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# =========================== INFERENCE TEST ===========================
TEST_PROMPT = "What was it like in the garden in Milan, the moment everything changed?"

# ============================================================================
print("✓ Configuration loaded (Augustine Qwen3 14B 4-bit QLoRA)")
print(f"  Base model: {BASE_LLM}")
print(f"  Model name: {MODEL_NAME_BASE}")
print(f"  Input data: {INPUT_DATA_FILE}")
print(f"  Output base: {OUTPUT_BASE_DIR}")
print(f"  LoRA output: {LORA_OUTPUT_DIR}")
print(f"  LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"  Training: batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LEARNING_RATE}")
print(f"  Training precision: 4-bit QLoRA")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")
print(f"  Voice modes: {list(VOICE_MODES.keys())}")
print(f"  Voice mode prompts: extracted from JSONL at load time")


✓ Configuration loaded (Augustine Qwen3 14B 4-bit QLoRA)
  Base model: unsloth/Qwen3-14B-unsloth-bnb-4bit
  Model name: augustine_qwen3_14b_unsloth_4bit
  Input data: /home/spark/projects/training/biblical/data/training-data/augustine_persona/augustine_sharegpt.jsonl
  Output base: /home/spark/projects/training/biblical/output/augustine_qwen3_14b_unsloth_4bit
  LoRA output: /home/spark/projects/training/biblical/output/augustine_qwen3_14b_unsloth_4bit/lora_adapters
  LoRA config: r=16, alpha=32, targets=['q_proj', 'k_proj', 'v_proj', 'o_proj']
  Training: batch=2, grad_accum=4, lr=0.0001
  Training precision: 4-bit QLoRA
  Max seq length: 4096
  Voice modes: ['confessional', 'philosophical', 'polemical', 'devotional']
  Voice mode prompts: extracted from JSONL at load time


## 2. Environment Preparation

Install Unsloth and updated HuggingFace libraries.

In [11]:
# Install core packages from PyPI (much faster than git installs)
!pip install -q unsloth transformers trl peft accelerate datasets bitsandbytes

# Verify installations
import unsloth
import transformers
import trl
print(f"\u2713 Unsloth: {unsloth.__version__}")
print(f"\u2713 Transformers: {transformers.__version__}")
print(f"\u2713 TRL: {trl.__version__}")
print("Environment ready!")

✓ Unsloth: 2026.2.1
✓ Transformers: 4.57.3
✓ TRL: 0.24.0
Environment ready!


## 3. Load Dataset

Load the Augustine ShareGPT JSONL from `biblical_datagen_augustine.ipynb`.

- Already quality-filtered (no short answers, no AI refusals, template contamination gated)
- Multi-turn: 4 QA pairs grouped per conversation by topic
- Each conversation has a **voice-mode-specific** system prompt (confessional/philosophical/polemical/devotional)
- Standard ShareGPT format: `[system, human, gpt, human, gpt, ...]`

In [12]:

import json, os
from collections import defaultdict
from datasets import Dataset as HFDataset

print(f"LOADING AUGUSTINE SHAREGPT DATA")
print(f"  File: {INPUT_DATA_FILE}")

# Load multi-turn conversations and EXTRACT system prompts from the JSONL.
# This replaces any hardcoded prompts — prompts stay in sync with datagen automatically.
conversations = []
voice_mode_system_prompts = {}   # mode_name -> full system prompt text
mode_counts = defaultdict(int)

with open(INPUT_DATA_FILE) as f:
    for line in f:
        conv = json.loads(line)
        conversations.append(conv)
        # Detect voice mode and extract system prompt
        sys_msg = conv["conversations"][0]["value"]
        detected = "unknown"
        for mode_name, mode_info in VOICE_MODES.items():
            if mode_info["detection_phrase"] in sys_msg:
                detected = mode_name
                if mode_name not in voice_mode_system_prompts:
                    voice_mode_system_prompts[mode_name] = sys_msg
                break
        mode_counts[detected] += 1

dataset = HFDataset.from_list(conversations)

print(f"\n{'='*50}")
print(f"Total dataset: {len(dataset)} multi-turn conversations across {len(mode_counts)} voice modes")
print(f"Extracted {len(voice_mode_system_prompts)} voice mode system prompts from JSONL")
print(f"Columns: {dataset.column_names}")
print(f"\nPer-voice-mode breakdown:")
for m, c in sorted(mode_counts.items(), key=lambda x: -x[1]):
    print(f"  {m:20s} {c:>5d} conversations  ({c/len(dataset)*100:.1f}%)")

# Show a sample extracted prompt
if voice_mode_system_prompts:
    sample_mode = next(iter(voice_mode_system_prompts))
    print(f"\n--- Sample extracted prompt ({sample_mode}, first 250 chars) ---")
    print(f"  {voice_mode_system_prompts[sample_mode][:250]}...")


LOADING AUGUSTINE SHAREGPT DATA
  File: /home/spark/projects/training/biblical/data/training-data/augustine_persona/augustine_sharegpt.jsonl

Total dataset: 2726 multi-turn conversations across 4 voice modes
Extracted 4 voice mode system prompts from JSONL
Columns: ['conversations']

Per-voice-mode breakdown:
  confessional          1947 conversations  (71.4%)
  philosophical          385 conversations  (14.1%)
  devotional             200 conversations  (7.3%)
  polemical              194 conversations  (7.1%)

--- Sample extracted prompt (confessional, first 250 chars) ---
  You are Saint Augustine of Hippo — Bishop, Doctor of the Church, convert from Manichaeism and worldly ambition, author of the Confessions and the City of God, a man who wrestled with sin, philosophy, and the overwhelming grace of God.

CURRENT VOICE ...


## 4. Validate & Summarize Dataset

Datagen data is already clean (quality-gated for template contamination). Verify structure and show voice mode distribution.

In [13]:

bad_examples = []
empty_responses = []
unique_system_prompts = set()

for i, example in enumerate(dataset):
    convs = example["conversations"]
    # Multi-turn ShareGPT: system, then alternating human/gpt pairs
    if len(convs) < 3 or len(convs) % 2 == 0:
        bad_examples.append((i, f"Expected odd turn count ≥3, got {len(convs)}"))
        continue
    if convs[0]["from"] != "system":
        bad_examples.append((i, f"First turn should be 'system', got '{convs[0]['from']}'"))
        continue
    # Validate alternating human/gpt after system
    role_ok = True
    for j in range(1, len(convs)):
        expected = "human" if j % 2 == 1 else "gpt"
        if convs[j]["from"] != expected:
            bad_examples.append((i, f"Turn {j} should be '{expected}', got '{convs[j]['from']}'"))
            role_ok = False
            break
    if not role_ok:
        continue
    # Check last GPT response is not empty
    if len(convs[-1]["value"].strip()) == 0:
        empty_responses.append(i)
    unique_system_prompts.add(convs[0]["value"])

# Turn-count distribution
from collections import Counter
turn_dist = Counter(len(ex["conversations"]) for ex in dataset)

print("DATA QUALITY CHECK")
print(f"  Total examples: {len(dataset)}")
print(f"  Bad structure: {len(bad_examples)}")
print(f"  Empty responses: {len(empty_responses)}")
print(f"  Unique system prompts: {len(unique_system_prompts)} (expected: ~{len(voice_mode_system_prompts)} voice modes)")
print(f"  Turn distribution: {dict(sorted(turn_dist.items()))}")

if bad_examples:
    print(f"\n⚠️ Bad examples (first 5):")
    for idx, reason in bad_examples[:5]:
        print(f"    Example {idx}: {reason}")

if empty_responses:
    print(f"\n⚠️ Filtering {len(empty_responses)} empty responses...")
    good_indices = [i for i in range(len(dataset)) if i not in set(empty_responses)]
    dataset = dataset.select(good_indices)
    print(f"  Dataset after filtering: {len(dataset)} examples")

# Voice mode distribution with balance check
print(f"\nVOICE MODE DISTRIBUTION:")
for mode, count in sorted(mode_counts.items(), key=lambda x: -x[1]):
    bar = "█" * (count // 20) + "▌" * (1 if count % 20 >= 10 else 0)
    print(f"  {mode:<16} {count:>5}  {bar}")
print(f"  {'TOTAL':<16} {sum(mode_counts.values()):>5}")

# Class imbalance check
known_counts = [c for m, c in mode_counts.items() if m != "unknown"]
if known_counts:
    smallest, largest = min(known_counts), max(known_counts)
    if smallest < largest * 0.15:
        print(f"\n⚠️ CLASS IMBALANCE: smallest mode ({smallest}) is <15% of largest ({largest})")
        print("  Consider upsampling underrepresented modes or generating more data.")
    else:
        print(f"\n✓ Balance OK: smallest mode is {smallest/largest*100:.0f}% of largest")

# Show voice differentiation — one sample from each mode
print(f"\nVOICE SAMPLES (first ~120 chars of response per mode):")
seen_modes = set()
for example in dataset:
    sys_msg = example["conversations"][0]["value"]
    for mode_name, mode_info in VOICE_MODES.items():
        if mode_info["detection_phrase"] in sys_msg and mode_name not in seen_modes:
            response_start = example["conversations"][2]["value"][:120]
            print(f"  [{mode_name:14s}] \"{response_start}...\"")
            seen_modes.add(mode_name)
            break
    if len(seen_modes) >= len(VOICE_MODES):
        break

print(f"\n✓ Dataset validated and ready for training")


DATA QUALITY CHECK
  Total examples: 2726
  Bad structure: 0
  Empty responses: 0
  Unique system prompts: 4 (expected: ~4 voice modes)
  Turn distribution: {5: 77, 7: 603, 9: 2046}

VOICE MODE DISTRIBUTION:
  confessional      1947  █████████████████████████████████████████████████████████████████████████████████████████████████
  philosophical      385  ███████████████████
  devotional         200  ██████████
  polemical          194  █████████▌
  TOTAL             2726

⚠️ CLASS IMBALANCE: smallest mode (194) is <15% of largest (1947)
  Consider upsampling underrepresented modes or generating more data.

VOICE SAMPLES (first ~120 chars of response per mode):
  [confessional  ] "How many nights did I lie awake, tormented by the illusion that my disordered loves were philosophy, when all along I wa..."
  [devotional    ] "Reason said to me in that hushed chamber of the soul where God is overheard rather than heard: *Can a man offer what he ..."
  [philosophical ] "Justice, that most b

## 5. Load Model & Tokenizer (4-bit)

Load Qwen3 14B in 4-bit precision for QLoRA training.

- **DGX Spark (128GB):** 14B 4-bit ~8GB — plenty of headroom
- **A5000 (24GB):** fits with room for KV-cache via vLLM

In [14]:
from unsloth import FastLanguageModel
import torch

# Load model in 4-bit precision for QLoRA training
# Use device_map={"":  0} to force all layers onto GPU 0
# DGX Spark has 119.7GB \u2014 plenty for 14B 4-bit (~8GB)
# device_map="auto" incorrectly offloads some layers to CPU which BnB 4-bit rejects
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # Auto-detect (will use bfloat16 for compute)
    load_in_4bit=True,    # 4-bit quantization for QLoRA
    device_map={"":  0}    # Force all layers onto GPU 0 (119GB unified memory)
)

# Only set pad_token if not already defined by the model
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"\u2713 Model loaded: {BASE_LLM}")
print(f"\u2713 Precision: 4-bit (QLoRA)")

print(f"\u2713 Pad token: {repr(tokenizer.pad_token)}")
print(f"\u2713 Max sequence length: {MAX_SEQ_LENGTH}")

==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.69 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 3/3 [00:58<00:00, 19.54s/it]


✓ Model loaded: unsloth/Qwen3-14B-unsloth-bnb-4bit
✓ Precision: 4-bit (QLoRA)
✓ Pad token: '<|vision_pad|>'
✓ Max sequence length: 4096


## 6. Format Dataset for Chat Template

Apply Qwen3's ChatML template to each conversation and build the final training dataset.

In [15]:
# Format dataset for Qwen3 chat template (ChatML) using Unsloth's standardize_sharegpt
from unsloth.chat_templates import standardize_sharegpt

dataset = standardize_sharegpt(dataset)
formatted_texts = tokenizer.apply_chat_template(
    list(dataset["conversations"]),
    tokenize=False,
)

# Build final dataset
import pandas as pd
from datasets import Dataset as HFDataset
dataset = HFDataset.from_pandas(pd.DataFrame({"text": formatted_texts}))

# Filter out empty examples and shuffle
dataset = dataset.filter(lambda x: len(x["text"]) > 0)
dataset = dataset.shuffle(seed=42)

print(f"--- Sample formatted text (first 500 chars) ---")
print(dataset[0]['text'][:500])
print(f"\n\u2713 Dataset formatted: {len(dataset)} examples")

Unsloth: Standardizing formats (num_proc=24):   0%|          | 0/2726 [00:00<?, ? examples/s]

Filter: 100%|██████████| 2726/2726 [00:00<00:00, 147504.61 examples/s]

--- Sample formatted text (first 500 chars) ---
<|im_start|>system
You are Saint Augustine of Hippo — Bishop, Doctor of the Church, convert from Manichaeism and worldly ambition, author of the Confessions and the City of God, a man who wrestled with sin, philosophy, and the overwhelming grace of God.

CURRENT VOICE MODE: Speaking from the Confessions — raw autobiographical honesty, prayer addressed directly to God, the journey from sin to grace

YOUR DISTINCTIVE VOICE IN THIS MODE: Intimate, prayerful, confessional. Addresses God directly in 

✓ Dataset formatted: 2726 examples


## 7. Add LoRA Adapters

Configure LoRA for efficient fine-tuning. See Step 1 config for module options.

In [16]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH
)

print(f"\u2713 LoRA adapters added (r={LORA_R}, targets={LORA_TARGET_MODULES})")
print(f"\u2713 Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

✓ LoRA adapters added (r=16, targets=['q_proj', 'k_proj', 'v_proj', 'o_proj'])
✓ Trainable parameters: 20,971,520


## 8. Trainer Setup

- 1 epoch, low learning rate (1e-4) — teaches Augustine's four voice registers without degrading base capabilities
- Each example has a voice-mode-specific system prompt so the model learns distinct registers

In [ ]:
from trl import SFTTrainer, SFTConfig

# With packing=True, TRL concatenates multiple conversations into each max_seq_length
# sequence. This means 1 epoch over the packed dataset has FEWER steps than the raw
# dataset count would suggest. We use num_train_epochs instead of max_steps so TRL
# calculates the correct step count automatically.
effective_batch_size = BATCH_SIZE * GRAD_ACCUM
raw_steps = len(dataset) // effective_batch_size
warmup_steps = max(1, raw_steps // 10)
save_steps = min(100, raw_steps)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,              # Pack multiple conversations per sequence — biggest speedup
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=warmup_steps,
        num_train_epochs=TARGET_EPOCHS,   # Let TRL compute steps for packed dataset (NOT max_steps)
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        report_to="none",
        save_strategy="steps",
        save_steps=save_steps,
    ),
)

# Report actual step count from trainer (accounts for packing)
actual_steps = trainer.state.max_steps if hasattr(trainer.state, 'max_steps') else "TBD (computed at train time)"
print("✓ Trainer configured for Augustine 14B 4-bit QLoRA training")
print(f"✓ Dataset size: {len(dataset)} conversations (packed into fewer dense sequences)")
print(f"✓ Packing: ENABLED — TRL computes actual epoch length from packed data")
print(f"✓ Effective batch size: {effective_batch_size} (batch={BATCH_SIZE} × grad_accum={GRAD_ACCUM})")
print(f"✓ Epochs: {TARGET_EPOCHS}")
print(f"✓ Warmup steps: {warmup_steps}")
print(f"✓ Save every: {save_steps} steps")
print(f"✓ Learning rate: {LEARNING_RATE}")


Unsloth: Packing train dataset (num_proc=24): 100%|██████████| 2726/2726 [00:01<00:00, 1897.89 examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
✓ Trainer configured for Augustine 14B 4-bit QLoRA training
✓ Dataset size: 2726 conversations (packed into fewer dense sequences)
✓ Packing: ENABLED — TRL computes actual epoch length from packed data
✓ Effective batch size: 8 (batch=2 × grad_accum=4)
✓ Epochs: 1
✓ Warmup steps: 34
✓ Save every: 100 steps
✓ Learning rate: 0.0001


## 9. Train

In [ ]:
# Start training
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,900 | Num Epochs = 1 | Total steps = 238
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,971,520 of 14,789,278,720 (0.14% trained)


## 10. Save LoRA Adapters

Save the trained LoRA adapters. These can be loaded on any Qwen3 14B model with PEFT or served directly via vLLM.

In [ ]:

from pathlib import Path
import json

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Save LoRA adapters + tokenizer
print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

# Save voice mode system prompts alongside adapters for inference use
prompts_path = f"{LORA_OUTPUT_DIR}/voice_mode_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(voice_mode_system_prompts, f, indent=2)

print(f"\n✓ LoRA adapters saved!")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  System prompts: {prompts_path} ({len(voice_mode_system_prompts)} voice modes)")
print(f"\n  At inference, load prompts with:")
print(f'    with open("{prompts_path}") as f:')
print(f'        prompts = json.load(f)')
print(f'    system_msg = prompts["confessional"]  # or philosophical, polemical, devotional')


## 11. Test Inference Across Voice Modes

The key validation: does Augustine sound **different** depending on voice mode? We test all four modes with voice-appropriate questions. The system prompts are pulled from the first matching conversation in the training data — the same prompts the model was trained on.

Confessional should sound prayerful, philosophical argumentative, polemical sharp, devotional contemplative.

In [ ]:

FastLanguageModel.for_inference(model)

# Use the system prompts already extracted at load time (voice_mode_system_prompts)
for mode_name, mode_info in VOICE_MODES.items():
    system_prompt = voice_mode_system_prompts.get(mode_name, "You are Saint Augustine of Hippo.")
    test_prompt = mode_info["test_prompt"]

    test_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": test_prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        [test_messages],
        tokenize=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    print(f"{'='*60}")
    print(f"  AUGUSTINE — {mode_name.upper()} MODE")
    print(f"  {mode_info['description']}")
    print(f"{'='*60}")
    print(f"Q: {test_prompt}")
    print(f"\nA: {response}")
    print()


## 12. Verify Saved Adapter

Load the saved adapter fresh from disk to confirm it produces coherent output when loaded independently.

In [ ]:

print(f"Verifying saved adapter from {LORA_OUTPUT_DIR}...")

verify_model, verify_tokenizer = FastLanguageModel.from_pretrained(
    LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    device_map={"": 0},
)

FastLanguageModel.for_inference(verify_model)

# Test with confessional mode — the most distinctive register
system_prompt = voice_mode_system_prompts.get("confessional", "You are Saint Augustine of Hippo.")
test_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": TEST_PROMPT},
]

inputs = verify_tokenizer.apply_chat_template(
    [test_messages],
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(verify_model.device)

outputs = verify_model.generate(
    input_ids=inputs,
    max_new_tokens=128,
    temperature=0.7,
    do_sample=True,
)

response = verify_tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(f"\nAUGUSTINE (confessional): {response}")
print(f"\n✓ Adapter verified!")

del verify_model, verify_tokenizer
torch.cuda.empty_cache()
print("✓ Verification model cleared from memory")
